# Part 1 — Policy Gradient Training (Colab)

Runs on a **GPU runtime**. Make sure you have selected:


In [ ]:
# ── Step 1: Check GPU ──────────────────────────────────────────────────────
import tensorflow as tf
print("GPUs available:", tf.config.list_physical_devices("GPU"))
print("TF version:", tf.__version__)

In [ ]:
# ── Step 2: Upload required files ──────────────────────────────────────────
# Run this cell — it will open a file picker.
# Upload ALL of these files:
#   connect4_env.py
#   loader.py                  (from models/loader.py)
#   josh_cnn.h5                (from models/josh_cnn.h5)
#   m1_iter100.keras           (from pg_checkpoints/)
#   m1_iter200.keras           (from pg_checkpoints/ — most trained, becomes M1)

from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
# ── Step 3: Wire up folders so imports work ────────────────────────────────
import os, shutil

os.makedirs("models", exist_ok=True)
os.makedirs("checkpoints/pg", exist_ok=True)

if os.path.exists("loader.py"):
    shutil.move("loader.py", "models/loader.py")
if os.path.exists("josh_cnn.h5"):
    shutil.move("josh_cnn.h5", "models/josh_cnn.h5")

# Move checkpoints into place
for ckpt in ["m1_iter100.keras", "m1_iter200.keras"]:
    if os.path.exists(ckpt):
        shutil.move(ckpt, f"checkpoints/pg/{ckpt}")

with open("models/__init__.py", "w") as f:
    f.write("")

print("models/:", os.listdir("models"))
print("checkpoints/pg/:", os.listdir("checkpoints/pg"))

In [ ]:
# ── Step 4: Hyperparameters — tune these ───────────────────────────────────
GAMMA             = 0.99
LR                = 1e-4
GAMES_PER_ITER    = 10
BATCH_SIZE        = 64     # keep fixed — TF retraces graph if this changes
RANDOM_INIT_MOVES = 3
ENTROPY_COEF      = 0.01
GRAD_CLIP         = 1.0
ADD_TO_POOL_EVERY = 100
MAX_OWN_SNAPSHOTS = 6
N_ITERATIONS      = 2000   # increase for tournament — Colab can handle more
EVAL_EVERY        = 50
EVAL_GAMES        = 100
CHECKPOINT_DIR    = "checkpoints/pg/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# ── Step 5: Imports ────────────────────────────────────────────────────────
import random, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from connect4_env import (
    encode_board, legal_moves, game_over, make_board, step,
    ModelAgent, RandomAgent, StrongRuleAgent, evaluate_agents
)
from models.loader import load_model

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("Imports OK")

In [ ]:
# ── Step 6: Load M1 (most trained checkpoint) ──────────────────────────────
# Using m1_iter200.keras as M1 — picks up training where it left off
# instead of starting from the original Project 1 CNN.

M1_CHECKPOINT = "checkpoints/pg/m1_iter200.keras"

m1_model  = keras.models.load_model(M1_CHECKPOINT, compile=False)
optimizer = keras.optimizers.Adam(learning_rate=LR)
print(f"Loaded M1 from {M1_CHECKPOINT}")
m1_model.summary()

In [ ]:
# ── Step 7: Build opponent pool ─────────────────────────────────────────────
# Pool includes:
#   - StrongRuleAgent    — rule-based baseline (always stays)
#   - RandomAgent        — easy warm-up (always stays)
#   - original josh_cnn  — Project 1 baseline (never removed per spec)
#   - m1_iter100         — early PG snapshot
#   - m1_iter200         — best PG snapshot so far (copy, not M1 itself)

from models.loader import load_model

def _make_m2_agent(model):
    return ModelAgent(model, sample=True, strong=True)

opponent_pool = [
    StrongRuleAgent(),
    RandomAgent(),
    _make_m2_agent(load_model("josh_cnn")),                                          # original — never removed
    _make_m2_agent(keras.models.load_model("checkpoints/pg/m1_iter100.keras", compile=False)),
    _make_m2_agent(keras.models.load_model("checkpoints/pg/m1_iter200.keras", compile=False)),
]
own_snapshot_count = 2   # we already have 2 of our own snapshots in the pool
print(f"Initial pool size: {len(opponent_pool)}")

In [ ]:
# ── Step 8: Helper functions ────────────────────────────────────────────────

def collect_episode(m1_player_id, m2_agent):
    m1_agent = ModelAgent(m1_model, sample=True, strong=False)
    agent1, agent2 = (m1_agent, m2_agent) if m1_player_id == 1 else (m2_agent, m1_agent)
    board = make_board(); current = 1; m1_trans = []

    for _ in range(RANDOM_INIT_MOVES):
        cols = legal_moves(board)
        if not cols: break
        board, _ = step(board, np.random.choice(cols), current)
        done, _ = game_over(board)
        if done: return [], 0
        current = -current

    while True:
        cols = legal_moves(board)
        if not cols: return m1_trans, 0
        if current == m1_player_id:
            enc = encode_board(board, current)
            lmask = np.zeros(7, dtype=np.float32); lmask[cols] = 1.
            col = m1_agent.select_move(board, current)
            m1_trans.append((enc, col, lmask))
        else:
            col = m2_agent.select_move(board, current)
        board, _ = step(board, col, current)
        done, winner = game_over(board)
        if done:
            outcome = 1 if winner == m1_player_id else (-1 if winner != 0 else 0)
            return m1_trans, outcome
        current = -current


def compute_discounted_returns(transitions, outcome, gamma=GAMMA):
    T = len(transitions)
    return [(enc, col, outcome * (gamma ** (T - 1 - t)), lmask)
            for t, (enc, col, lmask) in enumerate(transitions)]


@tf.function
def train_step(boards, cols, returns):
    with tf.GradientTape() as tape:
        probs        = m1_model(boards, training=True)
        log_probs    = tf.math.log(probs + 1e-8)
        indices      = tf.stack([tf.range(BATCH_SIZE, dtype=tf.int32), cols], axis=1)
        chosen       = tf.gather_nd(log_probs, indices)
        entropy      = -tf.reduce_mean(tf.reduce_sum(probs * log_probs, axis=1))
        loss         = -tf.reduce_mean(chosen * returns) - ENTROPY_COEF * entropy
    grads, _ = tf.clip_by_global_norm(
        tape.gradient(loss, m1_model.trainable_variables), GRAD_CLIP)
    optimizer.apply_gradients(zip(grads, m1_model.trainable_variables))
    return loss


def sample_batch(all_triplets):
    idx    = np.random.choice(len(all_triplets), size=BATCH_SIZE, replace=True)
    boards = np.stack([all_triplets[i][0] for i in idx]).astype(np.float32)
    cols   = np.array([all_triplets[i][1] for i in idx], dtype=np.int32)
    rets   = np.array([all_triplets[i][2] for i in idx], dtype=np.float32)
    if rets.std() > 1e-8:
        rets = (rets - rets.mean()) / (rets.std() + 1e-8)
    return tf.constant(boards), tf.constant(cols), tf.constant(rets)


print("Helpers defined")

In [ ]:
# ── Step 9: Training loop ───────────────────────────────────────────────────
log_loss, log_pg_loss = [], []
log_win_rate_strong, log_win_rate_orig, log_iters_eval = [], [], []

eval_strong  = StrongRuleAgent()
eval_orig_m1 = _make_m2_agent(load_model("josh_cnn"))

own_snapshot_count = 0

print(f"Training for {N_ITERATIONS} iterations...
")

for iteration in range(1, N_ITERATIONS + 1):

    m2_agent    = random.choice(opponent_pool)
    all_triplets = []

    for _ in range(GAMES_PER_ITER):
        m1_pid = random.choice([1, -1])
        trans, outcome = collect_episode(m1_pid, m2_agent)
        if not trans: continue
        all_triplets.extend(compute_discounted_returns(trans, outcome))

    if not all_triplets:
        continue

    boards_t, cols_t, rets_t = sample_batch(all_triplets)
    loss = train_step(boards_t, cols_t, rets_t)
    log_loss.append(float(loss))

    if iteration % EVAL_EVERY == 0:
        m1_eval = ModelAgent(m1_model, sample=False, strong=True)
        res_s   = evaluate_agents(m1_eval, eval_strong,  n_games=EVAL_GAMES)
        res_o   = evaluate_agents(m1_eval, eval_orig_m1, n_games=EVAL_GAMES)
        log_win_rate_strong.append(res_s["win_rate"])
        log_win_rate_orig.append(res_o["win_rate"])
        log_iters_eval.append(iteration)
        print(f"[{iteration:>5}] loss={float(loss):.4f} | "
              f"vs Strong: {res_s['win_rate']:.1%} | "
              f"vs OrigM1: {res_o['win_rate']:.1%}")

    if iteration % ADD_TO_POOL_EVERY == 0:
        snap = keras.models.clone_model(m1_model)
        snap.set_weights(m1_model.get_weights())
        opponent_pool.append(_make_m2_agent(snap))
        own_snapshot_count += 1
        if own_snapshot_count > MAX_OWN_SNAPSHOTS:
            opponent_pool.pop(3)
            own_snapshot_count -= 1
        ckpt = f"{CHECKPOINT_DIR}m1_iter{iteration}.keras"
        m1_model.save(ckpt)
        print(f"  → Snapshot saved. Pool size: {len(opponent_pool)}")

print("
Training complete!")

In [ ]:
# ── Step 10: Save final model ───────────────────────────────────────────────
final_path = CHECKPOINT_DIR + "m1_pg_final.keras"
m1_model.save(final_path)
print(f"Saved to {final_path}")

In [ ]:
# ── Step 11: Plot training curves ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Policy Gradient Training — M1 (josh_cnn)", fontsize=13)

window = 20
axes[0].plot(log_loss, alpha=0.3, label="Loss")
if len(log_loss) >= window:
    roll = np.convolve(log_loss, np.ones(window)/window, mode="valid")
    axes[0].plot(range(window-1, len(log_loss)), roll, label=f"MA{window}")
axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss"); axes[0].legend()

axes[1].plot(log_iters_eval, log_win_rate_strong, marker="o", label="vs Strong")
axes[1].plot(log_iters_eval, log_win_rate_orig,   marker="o", label="vs Orig M1")
axes[1].axhline(0.5, color="gray", linestyle="--", label="50% baseline")
axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("Win Rate")
axes[1].set_title("Win Rates"); axes[1].legend(); axes[1].set_ylim(0, 1)

plt.tight_layout()
plot_path = CHECKPOINT_DIR + "pg_training_curves.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Plot saved to {plot_path}")

In [ ]:
# ── Step 12: Download trained model ────────────────────────────────────────
from google.colab import files
files.download(CHECKPOINT_DIR + "m1_pg_final.keras")
files.download(CHECKPOINT_DIR + "pg_training_curves.png")